In [21]:
import os
from langgraph.graph import StateGraph, START, END 
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict
from dotenv import  load_dotenv

In [22]:
load_dotenv()

True

In [23]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Flash",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.7,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

In [24]:
chat_model = ChatHuggingFace(llm=llm)

In [25]:
# create a state 
class LLMState(TypedDict):
    question: str
    answer: str

In [26]:
def llm_qa(state:LLMState) -> LLMState:
    question = state["question"]

    #form a prompt
    prompt = f'Answer the following question {question}'

    answer = chat_model.invoke(prompt).content

    state["answer"] = answer

    return state

In [27]:
# create our graph 
graph = StateGraph(LLMState)

#add nodes
graph.add_node("llm_qa", llm_qa)


# add edge 
graph.add_edge(START, 'llm_qa')
graph.add_edge("llm_qa", END )


workflow = graph.compile()

In [30]:
que = {"question": "who is the president of the india"}

workflow.invoke(que)



{'question': 'who is the president of the india',
 'answer': 'The current President of India is **Droupadi Murmu**. She took office on July 25, 2022, and is the first person from the tribal community and the second woman to hold the office.'}